In [1]:
import os
import zipfile
import numpy as np
import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    GlobalAveragePooling1D,
    BatchNormalization,
    ReLU,
    Add,
    Dense,
    Multiply,
    Reshape,
    Activation
)

from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# 1. Seed
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 2. UCI-HAR Path
# ============================================================

DATA_PATH = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "UCI HAR Dataset"
)


# ============================================================
# 3. Download
# ============================================================

if not os.path.exists(DATA_PATH):

    print("Downloading UCI-HAR dataset...")

    url = (
        "https://archive.ics.uci.edu/ml/"
        "machine-learning-databases/00240/"
        "UCI%20HAR%20Dataset.zip"
    )

    zip_path = tf.keras.utils.get_file(
        "UCI_HAR.zip",
        origin=url,
        cache_dir="/content",
        cache_subdir=""
    )

    extract_path = (
        "/content/drive/MyDrive/Colab Notebooks"
    )

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_path)

else:
    print("Existing dataset found")


# ============================================================
# 4. Signal Names
# ============================================================

signals = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z"
]


# ============================================================
# 5. Load Dataset
# ============================================================

def load_data(split):

    X = []

    for signal in signals:

        file_path = (
            f"{DATA_PATH}/{split}/"
            f"Inertial Signals/"
            f"{signal}_{split}.txt"
        )

        data = np.loadtxt(file_path)
        X.append(data)

    # (N, 128, 9)
    X = np.stack(X, axis=-1)

    y_path = (
        f"{DATA_PATH}/{split}/y_{split}.txt"
    )

    y = np.loadtxt(y_path).astype(int)

    # Labels: 1~6 -> 0~5
    y = y - 1

    return X.astype(np.float32), y


X_train, y_train = load_data("train")
X_test, y_test = load_data("test")

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)


# ============================================================
# 6. Normalization
# ============================================================

mean = X_train.mean(
    axis=(0, 1),
    keepdims=True
)

std = X_train.std(
    axis=(0, 1),
    keepdims=True
)

X_train = (
    X_train - mean
) / (
    std + 1e-8
)

X_test = (
    X_test - mean
) / (
    std + 1e-8
)


# ============================================================
# 7. One-hot Encoding
# ============================================================

n_outputs = 6

y_train_onehot = to_categorical(
    y_train,
    num_classes=n_outputs
)

y_test_onehot = to_categorical(
    y_test,
    num_classes=n_outputs
)


# ============================================================
# 8. Squeeze-and-Excitation (SE) Block
# ============================================================

def se_block(x, reduction_ratio=8):

    channels = int(x.shape[-1])

    hidden_units = max(
        channels // reduction_ratio,
        1
    )

    # --------------------------------------------------------
    # Squeeze: Global Average Pooling
    # --------------------------------------------------------

    se = GlobalAveragePooling1D()(x)

    # --------------------------------------------------------
    # Excitation: Bottleneck Dense Layers
    # --------------------------------------------------------

    se = Dense(
        hidden_units,
        activation="relu",
        use_bias=True
    )(se)

    se = Dense(
        channels,
        activation="sigmoid",
        use_bias=True
    )(se)

    # (batch, channels) -> (batch, 1, channels)
    se = Reshape((1, channels))(se)

    # --------------------------------------------------------
    # Scale: Channel-wise Recalibration
    # --------------------------------------------------------

    out = Multiply()([
        x,
        se
    ])

    return out


# ============================================================
# 9. SE-ResNet Residual Block
# ============================================================

def se_residual_block(
    x,
    filters,
    kernel_size=3,
    reduction_ratio=8,
    downsample=False
):

    shortcut = x

    # --------------------------------------------------------
    # Main Path: First Conv1D
    # --------------------------------------------------------

    stride = 2 if downsample else 1

    x = Conv1D(
        filters,
        kernel_size=kernel_size,
        strides=stride,
        padding="same",
        use_bias=False
    )(x)

    x = BatchNormalization()(x)
    x = ReLU()(x)

    # --------------------------------------------------------
    # Main Path: Second Conv1D
    # --------------------------------------------------------

    x = Conv1D(
        filters,
        kernel_size=kernel_size,
        strides=1,
        padding="same",
        use_bias=False
    )(x)

    x = BatchNormalization()(x)

    # --------------------------------------------------------
    # Shortcut Projection
    # --------------------------------------------------------

    if (
        int(shortcut.shape[-1]) != filters
        or downsample
    ):

        shortcut = Conv1D(
            filters,
            kernel_size=1,
            strides=stride,
            padding="same",
            use_bias=False
        )(shortcut)

        shortcut = BatchNormalization()(shortcut)

    # --------------------------------------------------------
    # Residual Addition
    # --------------------------------------------------------

    x = Add()([
        x,
        shortcut
    ])

    x = ReLU()(x)

    # --------------------------------------------------------
    # Squeeze-and-Excitation
    # --------------------------------------------------------

    x = se_block(
        x,
        reduction_ratio=reduction_ratio
    )

    return x


# ============================================================
# 10. Build SE-ResNet1D Model
# ============================================================

inputs = Input(
    shape=(128, 9),
    name="input_signal"
)


# ------------------------------------------------------------
# Initial Convolution
# ------------------------------------------------------------

x = Conv1D(
    32,
    kernel_size=3,
    strides=1,
    padding="same",
    use_bias=False
)(inputs)

x = BatchNormalization()(x)
x = ReLU()(x)


# ------------------------------------------------------------
# Residual Block 1: 32 filters
# Output length: 128
# ------------------------------------------------------------

x = se_residual_block(
    x,
    filters=32,
    reduction_ratio=8
)


# ------------------------------------------------------------
# Downsampling
# Output length: 64
# ------------------------------------------------------------

x = MaxPooling1D(
    pool_size=2
)(x)


# ------------------------------------------------------------
# Residual Block 2: 64 filters
# Output length: 64
# ------------------------------------------------------------

x = se_residual_block(
    x,
    filters=64,
    reduction_ratio=8
)


# ------------------------------------------------------------
# Downsampling
# Output length: 32
# ------------------------------------------------------------

x = MaxPooling1D(
    pool_size=2
)(x)


# ------------------------------------------------------------
# Residual Block 3: 128 filters
# Output length: 32
# ------------------------------------------------------------

x = se_residual_block(
    x,
    filters=128,
    reduction_ratio=8
)


# ============================================================
# 11. Classification Head
# ============================================================

x = GlobalAveragePooling1D()(x)

x = Dense(
    128,
    activation="relu"
)(x)

outputs = Dense(
    n_outputs,
    activation="softmax",
    name="classification"
)(x)


model = Model(
    inputs=inputs,
    outputs=outputs,
    name="SE_ResNet1D_HAR"
)


# ============================================================
# 12. Compile
# ============================================================

model.compile(
    loss="categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    metrics=["accuracy"]
)

model.summary()


# ============================================================
# 13. Train
# ============================================================

history = model.fit(
    X_train,
    y_train_onehot,
    epochs=20,
    batch_size=64,
    validation_split=0.2,
    shuffle=True,
    verbose=1
)


# ============================================================
# 14. Prediction
# ============================================================

y_prob = model.predict(
    X_test,
    verbose=0
)

y_pred = np.argmax(
    y_prob,
    axis=1
)


# ============================================================
# 15. Evaluation
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)


print("\n==============================")
print("SE-RESNET1D TEST RESULTS")
print("==============================")

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")


# ============================================================
# 16. Classification Report
# ============================================================

class_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]

print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 17. Confusion Matrix
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\nConfusion Matrix")
print(cm)

Existing dataset found
X_train: (7352, 128, 9)
X_test : (2947, 128, 9)


Model: "SE_ResNet1D_HAR"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_signal        │ (None, 128, 9)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 128, 32)   │        864 │ input_signal[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 128, 32)   │        128 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 128, 32)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 128, 32)   │      3,072 │ re_lu[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 32)   │        128 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 128, 32)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 128, 32)   │      3,072 │ re_lu_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 32)   │        128 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 128, 32)   │          0 │ batch_normalizat… │
│                     │                   │            │ re_lu[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 128, 32)   │          0 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ re_lu_2[0][0]     │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 4)         │        132 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │        160 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 32)     │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 128, 32)   │          0 │ re_lu_2[0][0],    │
│                     │                   │            │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 64, 32)    │          0 │ multiply[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 64, 64)    │      6,144 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64)    │        256 │ conv1d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 64, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 135,010 (527.38 KB)

 Trainable params: 133,666 (522.13 KB)

 Non-trainable params: 1,344 (5.25 KB)

Epoch 1/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 41s 197ms/step - accuracy: 0.9002 - loss: 0.3056 - val_accuracy: 0.6621 - val_loss: 0.8813
Epoch 2/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.9522 - loss: 0.1112 - val_accuracy: 0.9123 - val_loss: 0.3577
Epoch 3/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9578 - loss: 0.0988 - val_accuracy: 0.8946 - val_loss: 0.3117
Epoch 4/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9612 - loss: 0.0913 - val_accuracy: 0.9041 - val_loss: 0.3539
Epoch 5/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9604 - loss: 0.0912 - val_accuracy: 0.9062 - val_loss: 0.3797
Epoch 6/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9626 - loss: 0.0859 - val_accuracy: 0.8790 - val_loss: 0.4259
Epoch 7/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9646 - loss: 0.0802 - val_accuracy: 0.8776 - val_loss: 0.3857
Epoch 8/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9604 - loss: 0.0927 - val_accuracy: 0.8804 - val